# DUPGT with semantics and retrieved LLM preferences

This notebook explains the two new code paths. The executed examples use **16 real Books and 16 real Movies**, encoded locally with Jina. This is a bounded code check, not a recommendation-accuracy experiment.

The full-data stages below are disabled by default. Enable them one stage at a time. Existing baseline weights, splits and the previous LLM pilot are preserved.

In [1]:
from pathlib import Path
import json
import sys
import subprocess
import numpy as np

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from dupgt_cdr.semantics import SemanticStore, FEATURE_NAMES

CHECK = ROOT / "data/semantics/amazon2023/encoding_check"
SEMANTICS = "data/semantics/amazon2023/jina_v2"
TEACHER = "data/teacher/amazon2023/retrieval_v1"
RUNS = ROOT / "outputs/amazon2023/semantic_transfer/full_seed2020"

def run(*arguments):
    subprocess.run([sys.executable, "-m", "dupgt_cdr", *arguments], cwd=ROOT, check=True)

print("Project:", ROOT)

Project: /Users/parjanyapandey/Programming/Masters/Data Mining/dupgt-cdr


## 1. Shared item meanings

The frozen encoder represents each title, description and category list as 768 numbers. Books and Movies use the same encoder and revisions. Padding and items without accepted text have zero vectors. The mapping connects compact item IDs to vector rows. The checksums prevent using a different catalogue or benchmark accidentally.

In [2]:
store = SemanticStore(CHECK)
print("Encoder:", store.manifest["encoder"]["model"])
print("Dimension:", store.vectors.shape[1])
print("Covered items:", store.manifest["counts"])
print("Scope:", store.manifest["catalogue_scope"])
print("Padding is zero:", bool(np.all(store.lookup([0]) == 0)))

Encoder: jinaai/jina-embeddings-v2-base-en
Dimension: 768
Covered items: {'book': 16, 'movie': 16}
Scope: bounded_trial
Padding is zero: True


## 2. Semantic retrieval

This example retrieves Movies using a Book vector directly, without an LLM. Cosine similarity measures content similarity; it is not a star rating or a probability of enjoyment. These neighbours are drawn only from the small check catalogue.

In [3]:
items = [json.loads(line) for line in (CHECK / "items.jsonl").read_text().splitlines()]
by_id = {row["item_id"]: row for row in items} if items and "item_id" in items[0] else {
    int(item): row for item, row in zip(store.ids[1:], items)
}
book = next(item for item in store.ids if 0 < item <= store.manifest["n_source"])
ids, similarities = store.retrieve(store.lookup([book])[0], top_k=5)
print("Book:", by_id[int(book)]["title"])
for item, similarity in zip(ids, similarities):
    print(f"{similarity:.3f}  {by_id[int(item)]['title']}")

Book: France: The Beautiful Cookbook- Authentic Recipes from the Regions of France
0.781  The Way To Cook DVD
0.706  Gifted Hands: The Ben Carson Story
0.702  A NATION ADRIFT A Chronicle of America's Providential Heritage
0.700  The Story Adult Curriculum DVDR: Getting to the Heart of God's Story
0.699  Sacred Marriage: What If God Designed Marriage To Make Us Holy More Than To Make Us Happy?


## 3. High/low semantic evidence

DUPGT retains twenty high-rating Book slots and twenty low-rating slots. We average the available text vectors separately, keeping the two signals distinct. The learned gate receives both profiles, the candidate Movie vector, and twelve evidence features. Coherence measures how closely the Book vectors agree; it is not calibrated confidence.

The following history is an **illustration assembled from catalogue IDs**, not a real user's ratings. A low rating does not tell us why that person disliked an item; DUPGT's 1–3 bucket also includes potentially neutral feedback.

In [4]:
books = store.ids[(store.ids > 0) & (store.ids <= store.manifest["n_source"])]
movie = int(ids[0])
x = np.zeros((1, 42), dtype=np.int64)
x[0, 1] = movie
x[0, 2:5] = books[:3]
x[0, 22:24] = books[3:5]
import torch
profiles, target, features, valid = store.features(torch.as_tensor(x))
print("High/low profile shape:", profiles.shape)
print("Available semantic evidence:", bool(valid[0]))
for name, value in zip(FEATURE_NAMES, features[0]):
    print(f"{name}: {value:.4f}")

High/low profile shape: (1, 2, 768)
Available semantic evidence: True
similarity_high: 0.7818
similarity_low: 0.7914
count_high: 0.4553
count_low: 0.3608
coverage_high: 1.0000
coverage_low: 1.0000
coherence_high: 0.9168
coherence_low: 0.9277
has_high: 1.0000
has_low: 1.0000
has_movie_text: 1.0000
similarity_difference: -0.0096


## 4. Encode the full accepted catalogue

`prepare_semantics` streams the cleaned, release-matched item records, encodes in small batches and saves progress after each batch. Repeating the same command resumes. It includes all accepted Book metadata and warm Movie metadata, without using hidden Movie ratings. Use MPS for encoding on this Mac, or CPU if needed. This step does not retrain DUPGT's domain factors.

In [5]:
RUN_FULL_ENCODING = False
if RUN_FULL_ENCODING:
    run("semantic-prepare", "--dataset", "2023", "--size", "full", "--device", "mps",
        "--output-dir", SEMANTICS)
else:
    print("Full catalogue encoding is disabled. The small encoding check is already saved.")

Full catalogue encoding is disabled. The small encoding check is already saved.


## 5. Generate source-only users' tentative Movie interests

`prepare_profiles` selects genuine source-only training users and records their actual Book ratings. Qwen receives Book content and ratings, with no Movie outcomes or original user IDs. It returns up to three descriptions of possible Movie interests, or abstains. These descriptions are not observed Movie interactions.

Load the verified Qwen model in LM Studio before enabling generation. First use a five-request trial and inspect its responses. Cached successful responses are reused. The token budget is cumulative; the request limit counts new attempts in each invocation.

In [6]:
PREPARE_TEACHER = False
GENERATE_INTERESTS = False
if PREPARE_TEACHER:
    run("profile-prepare", "--dataset", "2023", "--semantic-dir", SEMANTICS,
        "--teacher-users", "200", "--output-dir", TEACHER)
    run("profile-generate", "--teacher-dir", TEACHER, "--dry-run")
if GENERATE_INTERESTS:
    run("profile-generate", "--teacher-dir", TEACHER, "--llm-model", "dupgt-qwen35-9b",
        "--max-requests", "5", "--max-total-tokens", "100000")
if not PREPARE_TEACHER and not GENERATE_INTERESTS:
    print("Teacher preparation and generation are disabled.")

Teacher preparation and generation are disabled.


## 6. Retrieve actual Movies and construct weak supervision

After every requested response is cached, `build_retrieval` embeds the generated descriptions with the same Jina encoder, averages them, and retrieves ten real Movie IDs. Each retrieved item is compared with an unlabelled background Movie for a weak ranking loss. Background items are not known dislikes.

The direct-text control uses a normalized weighted sum of the same Book-text vectors, with weight equal to rating minus 3 (3 is neutral). The shuffled-profile control uses another user's generated interests. All use the same accepted users and pair budget; Movie endpoints can differ. This is inspired by LGCD's generation/retrieval stage, with mean pooling and pairwise distillation adapted for DUPGT. It does not implement LGCD's sequential encoder, diffusion or MoE.

In [7]:
RETRIEVE_MOVIES = False
if RETRIEVE_MOVIES:
    run("profile-retrieve", "--teacher-dir", TEACHER, "--device", "mps", "--top-k", "10")
else:
    print("Full teacher retrieval is disabled.")

Full teacher retrieval is disabled.


## 7. Train matching models

All variants reuse the same pretrained Book/Movie factors and start with fresh transfer modules. The semantic gate begins with a zero contribution, recovering DUPGT exactly. Rating MSE trains the gate and bridge; factors and cached text vectors remain frozen. There is no separate contrastive loss in this version.

The LLM variant adds `lambda * mean(softplus(-(score_A-score_B)))` to real-rating MSE. Each loss is averaged separately. Lambda is a training trade-off, not confidence. Semantics without a teacher supplies the zero-weight candidate.

This example uses 100% of the shared training users and one positive weight. Use the same settings for every variant; try positive weights 0.01 and 0.1 on validation. Set the fraction identically in every variant for any separate limited-supervision experiment. Full domain logs were still used for factor pretraining.

In [8]:
RUN_TRAINING = False
FRACTION = "1.0"
variants = [
    ("dupgt", None, "none", "0"),
    ("semantics", SEMANTICS, "none", "0"),
    ("semantics_llm", SEMANTICS, "retrieval_llm", "0.01"),
    ("semantics_text", SEMANTICS, "retrieval_text", "0.01"),
    ("semantics_shuffled", SEMANTICS, "retrieval_shuffled", "0.01"),
]
if RUN_TRAINING:
    for name, cache, teacher, weight in variants:
        arguments = ["transfer-train", "--dataset", "2023", "--device", "cpu",
                     "--overlap-fraction", FRACTION, "--pseudo-weight", weight,
                     "--teacher-kind", teacher, "--output-dir", str(RUNS / name)]
        if cache:
            arguments += ["--semantic-dir", cache]
        if teacher != "none":
            arguments += ["--teacher-dir", TEACHER]
        run(*arguments)
else:
    print("New model training is disabled. No accuracy result is implied by the code checks.")

New model training is disabled. No accuracy result is implied by the code checks.


## 8. Read validation results and reuse weights

Each run stores its best-validation checkpoint, training losses, settings, frozen-factor checks and validation metrics. `load_saved_model` reconstructs the correct architecture and verifies its semantic cache. Prediction requires saved weights and embeddings; it does not require another LLM call or retraining.

`transfer-compare` checks matching supervision and training budgets. Final evaluation requires an explicit immutable `transfer-lock` record followed by `transfer-evaluate`. The earlier pilot has already exposed the old test split: further development uses validation, and another result on that same test split is a follow-up experiment rather than untouched confirmation.

In [9]:
for name, _, _, _ in variants:
    path = RUNS / name / "validation_metrics.json"
    if path.exists():
        result = json.loads(path.read_text())
        print(name, result["raw"], "factors unchanged:", result["factors_unchanged"])
    else:
        print(name, "— not trained")

dupgt — not trained
semantics — not trained
semantics_llm — not trained
semantics_text — not trained
semantics_shuffled — not trained


The relevant source files are `semantics.py` (encoder, cache and semantic gate), `retrieval.py` (generated interests and catalogue retrieval), `experiments.py` (matched transfer training), and `runtime.py` (checkpoint loading). Both releases use these same functions with their own metadata, IDs and factor checkpoints.